# NB 6.4 &mdash; A espatlles de gegants

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT6** &mdash; Xarxes neuronals convolucionals i introducció a la visió per computador

---
### D'on venim

Al NB 6.2, amb 2.000 imatges no passàvem gaire del 50%, i cap medicina no ho arreglava: **faltaven dades**.
A la secció 11.2 del NB 1.2 ja vam avançar la solució: aprofitar una xarxa que uns altres ja han entrenat.

### Què farem avui

Farem servir una xarxa entrenada amb més d'un milió d'imatges, primer tal com és i després per
ensenyar-li coses noves **amb molt poques imatges i en menys d'un segon**. Al final la posarem a prova amb la
teva mà i amb fotos de satèl·lit.

### Preguntes que has de saber respondre en acabar

1. Què sap fer una xarxa preentrenada, i què respon davant d'una cosa que no ha vist mai?
2. Per què es pot reaprofitar una xarxa per a un problema que no és el seu?
3. Amb quantes imatges compensa el transfer learning, i a partir de quan una CNN pròpia l'atrapa?
4. Què passa quan les dades reals no s'assemblen a les de l'entrenament?

## 0. Preparació

Les biblioteques i la plantilla d'entrenament, com sempre.

In [ ]:
import time
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.neural_network import MLPClassifier
from torch.utils.data import TensorDataset

URL_UT = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT06-Xarxes_convolucionals/"
URL_RPS = URL_UT + "pedra_paper_tisora/rps.zip"
URL_EUROSAT = URL_UT + "eurosat/eurosat_1000.zip"

plt.rcParams["figure.figsize"] = (6, 4)

La **plantilla d'entrenament**, igual que abans. No cal modificar-la.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plantilla d'entrenament. No cal modificar-la: només cal saber què fa cada funció.
# Fa el mateix que MLPClassifier feia per dins a la UT5, però a la vista.
# ─────────────────────────────────────────────────────────────────────────────
import math
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # GPU si n'hi ha; si no, CPU


def make_cnn(in_channels, n_classes, image_size=28, filters=(16, 32), pooling=True,
             global_pool=False, batch_norm=False, dropout=0.0):
    """Crea una CNN a partir de paràmetres, com MLPClassifier amb hidden_layer_sizes.

    Paràmetres:
        in_channels: canals de la imatge (1 en gris, 3 en color).
        n_classes: quantes classes ha de distingir.
        image_size: costat de la imatge en píxels.
        filters: filtres de cada capa convolucional; (16, 32) són dues capes.
        pooling: si cada capa redueix la imatge a la meitat amb MaxPool2d.
        global_pool: si al final es fa la mitjana de cada mapa en lloc d'aplanar-los.
        batch_norm: si s'afegeix BatchNorm2d després de cada convolució.
        dropout: fracció de neurones que s'apaguen a l'atzar a l'entrenament.

    Retorna el model. Exemple: model = make_cnn(1, 10, filters=(16, 32, 64))
    """
    n_pools = (len(filters) - 1 if global_pool else len(filters)) if pooling else 0
    side = image_size // 2 ** n_pools
    if side < 1:
        raise ValueError(f"Hi ha massa capes amb pooling per a una imatge de {image_size} píxels: "
                         "fes filters més curt o prova pooling=False.")

    torch.manual_seed(SEED)
    layers = []
    channels = in_channels
    for position, n_filters in enumerate(filters):
        layers += [nn.Conv2d(channels, n_filters, kernel_size=3, padding=1)]
        if batch_norm:
            layers += [nn.BatchNorm2d(n_filters)]
        layers += [nn.ReLU()]
        is_last = position == len(filters) - 1
        if pooling and not (global_pool and is_last):
            layers += [nn.MaxPool2d(2)]
        channels = n_filters

    switch_off = [nn.Dropout(dropout)] if dropout > 0 else []
    if global_pool:
        layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), *switch_off, nn.Linear(channels, n_classes)]
    else:
        layers += [nn.Flatten(), nn.Linear(channels * side * side, 64), nn.ReLU(),
                   *switch_off, nn.Linear(64, n_classes)]
    return nn.Sequential(*layers)


def count_parameters(model):
    """Retorna quants pesos entrenables té el model."""
    return sum(weights.numel() for weights in model.parameters() if weights.requires_grad)


def make_loaders(train_set, val_set, batch_size=64, augment=None):
    """Prepara els mini-batches d'entrenament i de validació.

    Paràmetres:
        train_set, val_set: conjunts de dades de PyTorch.
        batch_size: imatges per mini-batch.
        augment: transformació per a cada imatge d'entrenament (data augmentation) o None.

    Retorna (train_loader, val_loader). Només l'entrenament es barreja i s'augmenta.
    """
    def collate_augmented(batch):
        images = torch.stack([augment(image) for image, _ in batch])
        labels = torch.as_tensor([label for _, label in batch])
        return images, labels

    collate = collate_augmented if augment is not None else None
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_set, batch_size=1000)
    return train_loader, val_loader


@torch.no_grad()
def _evaluate(model, loader):
    """Retorna la loss mitjana i l'encert del model amb un loader."""
    model.eval()
    device = next(model.parameters()).device
    loss_function = nn.CrossEntropyLoss(reduction="sum")
    total_loss, hits, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        scores = model(images)
        total_loss += loss_function(scores, labels).item()
        hits += (scores.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, hits / total


def accuracy(model, loader):
    """Retorna l'encert del model (de 0 a 1), com score() a scikit-learn."""
    return _evaluate(model, loader)[1]


def train_model(model, train_loader, val_loader, epochs, lr, optimizer="adam"):
    """Entrena el model i mostra com va, epoch a epoch. És el fit() d'aquesta unitat.

    Paràmetres:
        model: la xarxa, feta amb make_cnn.
        train_loader, val_loader: els mini-batches, fets amb make_loaders.
        epochs: quantes vegades es recorren totes les dades d'entrenament.
        lr: el learning rate.
        optimizer: "adam" o "sgd".

    Retorna un diccionari amb les llistes train_loss, val_loss, train_acc i val_acc.
    """
    optimizers = {"adam": torch.optim.Adam, "sgd": torch.optim.SGD}
    if optimizer not in optimizers:
        raise ValueError(f"Optimitzador desconegut: {optimizer!r}. Tria 'adam' o 'sgd'.")

    torch.manual_seed(SEED)
    model.to(DEVICE)
    step_maker = optimizers[optimizer](model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()                                  # activa dropout i batch norm
        total_loss, hits, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            step_maker.zero_grad()                     # esborra el repartiment anterior
            scores = model(images)                     # 1. la xarxa prediu
            loss = loss_function(scores, labels)       # 2. mesurem l'error (la loss)
            loss.backward()                            # 3. backpropagation: repartim la culpa
            step_maker.step()                          # 4. una passa de gradient descent
            total_loss += loss.item() * len(labels)
            hits += (scores.argmax(dim=1) == labels).sum().item()
            total += len(labels)

        val_loss, val_acc = _evaluate(model, val_loader)
        history["train_loss"].append(total_loss / total)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(hits / total)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch}/{epochs} · loss {total_loss / total:.3f} · "
              f"encert a validació {val_acc:.1%} · {time.perf_counter() - start:.0f} s")

        if not math.isfinite(total_loss):
            print("La loss s'ha disparat i l'entrenament s'atura: "
                  "segurament el learning rate és massa alt.")
            break
    return history


def plot_history(history):
    """Dibuixa les corbes de loss i d'encert d'un entrenament."""
    fig, grafics = plt.subplots(1, 2, figsize=(10, 3.5))
    epochs = range(1, len(history["train_loss"]) + 1)
    grafics[0].plot(epochs, history["train_loss"], "o-", label="entrenament")
    grafics[0].plot(epochs, history["val_loss"], "o-", label="validació")
    grafics[0].set_ylabel("loss")
    grafics[1].plot(epochs, history["train_acc"], "o-", label="entrenament")
    grafics[1].plot(epochs, history["val_acc"], "o-", label="validació")
    grafics[1].set_ylabel("encert")
    for grafic in grafics:
        grafic.set_xlabel("epoch")
        grafic.legend()
    plt.tight_layout()
    plt.show()

I les eines d'aquest notebook. Tampoc cal modificar-les.

In [ ]:
# Funcions per reaprofitar una xarxa ja entrenada i carregar fotos. No cal modificar-la.
import io
import zipfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from torch import nn
from torchvision import models
from torchvision.transforms import v2

WEIGHTS = models.ResNet18_Weights.IMAGENET1K_V1
_LOADED = {}


def _resnet():
    """Carrega ResNet18 una sola vegada: la primera crida baixa uns 45 MB de pesos."""
    if not _LOADED:
        model = models.resnet18(weights=WEIGHTS).eval()
        _LOADED["model"] = model
        _LOADED["backbone"] = nn.Sequential(*list(model.children())[:-1], nn.Flatten()).eval()
        _LOADED["prepare"] = WEIGHTS.transforms()
    return _LOADED


def _to_rgb(images):
    """Converteix imatges en gris a tres canals, que és el que espera ResNet18."""
    return images.repeat_interleave(3, dim=-3) if images.shape[-3] == 1 else images


@torch.no_grad()
def top3_imagenet(image):
    """Retorna les tres classes d'ImageNet més probables d'una imatge: [(nom en anglès, probabilitat), ...]."""
    resnet = _resnet()
    scores = resnet["model"](resnet["prepare"](_to_rgb(image).unsqueeze(0)))[0]
    values, positions = torch.softmax(scores, dim=0).topk(3)
    return [(WEIGHTS.meta["categories"][p], v) for p, v in zip(positions.tolist(), values.tolist())]


@torch.no_grad()
def extract_features(images, batch_size=32):
    """Passa les imatges per ResNet18, sense l'última capa, i retorna 512 números per imatge."""
    resnet = _resnet()
    pieces = [resnet["backbone"](resnet["prepare"](_to_rgb(batch)))
              for batch in torch.split(images, batch_size)]
    return torch.cat(pieces).numpy()


def _load_rgb(file, size):
    """Obre una imatge d'un fitxer, respecta l'orientació del mòbil i la passa a tensor RGB."""
    image = ImageOps.exif_transpose(Image.open(file)).convert("RGB")
    if size is not None:
        image = image.resize((size, size), Image.BILINEAR)
    return torch.tensor(np.array(image) / 255.0, dtype=torch.float32).permute(2, 0, 1)


def load_image_zip(url_or_path, size=224):
    """Carrega un zip amb carpetes train/<classe>/ i test/<classe>/.

    Retorna un diccionari amb les claus train i test: cadascuna (imatges, etiquetes, noms de classe).
    """
    if str(url_or_path).startswith(("http", "file:")):
        with urllib.request.urlopen(str(url_or_path), timeout=300) as response:
            content = response.read()
    else:
        content = Path(url_or_path).read_bytes()
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        files = sorted(name for name in archive.namelist() if name.lower().endswith((".jpg", ".jpeg", ".png")))
        class_names = sorted({name.split("/")[1] for name in files})
        data = {}
        for part in ["train", "test"]:
            chosen = [name for name in files if name.startswith(part + "/")]
            images = torch.stack([_load_rgb(io.BytesIO(archive.read(name)), size) for name in chosen])
            labels = torch.tensor([class_names.index(name.split("/")[1]) for name in chosen])
            data[part] = (images, labels, class_names)
    return data


def load_photo(path, size=224):
    """Carrega una foto com a tensor (3, alt, ample) amb valors de 0 a 1.

    Paràmetres:
        path: on és el fitxer.
        size: el costat de la imatge quadrada que es vol. None la deixa com és.
    """
    try:
        return _load_rgb(path, size)
    except UnidentifiedImageError as error:
        raise ValueError(f"No es pot obrir {Path(path).name}. Si és una foto HEIC d'iPhone, converteix-la "
                         "a JPG: obre-la a l'ordinador i exporta-la, o canvia la càmera a «Més compatible».") from error


def upload_photo():
    """Obre el selector de fitxers de Colab i retorna el nom de la foto, o None si no n'hi ha."""
    try:
        from google.colab import files
    except ImportError:
        print("Aquest botó només funciona a Colab. Aquí pots fer servir load_photo('ruta/de/la/foto.jpg').")
        return None
    uploaded = files.upload()
    return next(iter(uploaded), None)


def take_per_class(labels, n, seed=42):
    """Tria n imatges de cada classe a l'atzar i en retorna les posicions."""
    generator = torch.Generator().manual_seed(seed)
    chosen = []
    for label in labels.unique():
        positions = (labels == label).nonzero().flatten()
        if len(positions) < n:
            raise ValueError(f"La classe {label.item()} només té {len(positions)} imatges: tria un n més petit.")
        chosen.append(positions[torch.randperm(len(positions), generator=generator)[:n]])
    return torch.cat(chosen)


def predict_photo(mlp, image, class_names):
    """Passa una foto per ResNet18 i pel teu MLPClassifier. Retorna [(nom, probabilitat), ...], de més a menys."""
    probabilities = mlp.predict_proba(extract_features(image.unsqueeze(0)))[0]
    return sorted(zip(class_names, probabilities.tolist()), key=lambda pair: -pair[1])


def show_prediction(image, top):
    """Dibuixa una foto al costat de les probabilitats que hi dona el model."""
    fig, grafics = plt.subplots(1, 2, figsize=(7.5, 2.8))
    grafics[0].imshow(image.permute(1, 2, 0).squeeze().numpy(), cmap="gray")
    grafics[0].axis("off")
    grafics[1].barh([name for name, _ in top][::-1], [value for _, value in top][::-1], color="#1C7293")
    grafics[1].set_xlim(0, 1)
    grafics[1].set_xlabel("probabilitat")
    plt.tight_layout()
    plt.show()


def make_pretrained(n_classes, unfreeze=True):
    """Crea una ResNet18 amb una última capa nova, per fer fine-tuning amb train_model.

    Paràmetres:
        n_classes: quantes classes ha de distingir.
        unfreeze: si també es deixa aprendre l'últim bloc de capes (layer4).

    Fa servir imatges de 224 × 224 píxels i, amb CPU, va molt lenta: és per a la GPU.
    """
    torch.manual_seed(SEED)
    network = models.resnet18(weights=WEIGHTS)
    for weights in network.parameters():
        weights.requires_grad = False
    if unfreeze:
        for weights in network.layer4.parameters():
            weights.requires_grad = True
    network.fc = nn.Linear(network.fc.in_features, n_classes)
    return nn.Sequential(v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), network)

## 1. Una xarxa ja entrenada

**ImageNet** és un conjunt d'1,2 milions de fotos i **1.000 classes**, des de races de gos fins a
instruments musicals. Hi ha xarxes entrenades amb aquest conjunt durant dies, amb moltes GPU,
i tothom les pot baixar.

Farem servir **ResNet18**. Té 18 capes, i per això no es pot entrenar a classe: cada imatge ha de
passar per moltes capes i hi ha molts pesos. Però fer-la **servir** és molt més barat.

| | La teva CNN del NB 6.3 | ResNet18 |
|---|---|---|
| **Pesos** | 90.058 | 11.689.512 |
| **Imatges d'entrenament** | 40.000 | 1,2 milions |
| **Mida de cada imatge** | 32 × 32 | 224 × 224 |

La funció `top3_imagenet(image)` hi passa una imatge i en retorna les tres classes més probables, amb
els noms **en anglès**. La primera vegada baixa uns 45 MB de pesos. Baixem també tres fotos de mostra,
i `load_photo` les llegeix.

> ⏱ Cada foto, menys d'un segon.

In [ ]:
for name in ["gata.png", "cafe.png", "coet.jpg"]:
    urllib.request.urlretrieve(URL_UT + "fotos/" + name, name)

photo = load_photo("gata.png", size=None)
top = top3_imagenet(photo)
print("Pesos de ResNet18:", count_parameters(models.resnet18()))
show_prediction(photo, top)

**Egyptian cat (40%), tiger cat (38%) i tabby (22%):** tres mena de gat. ImageNet distingeix races, i per això la
confiança es reparteix. En realitat no s'equivoca gaire.

### Ara tu · Missió 1: què sap, i què no

1. **Aposta.** Què creus que dirà ResNet18 de `cafe.png`? I de `coet.jpg`? I d'una mà fent el gest de
   «pedra», si ImageNet no té cap classe «mà»?
2. **Experimenta.** Passa-li les dues fotos que falten amb `top3_imagenet` i `show_prediction`. Per a la mà, en
   tens una a la secció següent, però aquí la pots carregar amb `load_image_zip(URL_RPS)["train"][0][0]`.
3. **Foto pròpia (opcional).** Puja una foto d'un objecte amb `load_photo(upload_photo())`.
   **Norma de privacitat: només objectes, sense cares, persones, matrícules ni dades privades.** Els fitxers que
   puges a Colab queden a la seva màquina durant la sessió, i s'hi envien.
4. **Explica.** Què fa quan ha de respondre sobre una cosa que no és a les seves 1.000 classes? La probabilitat
   t'avisa que no ho sap? Recorda què passava al Pictionary del NB 6.2.

In [ ]:
# El teu codi aquí

## 2. Reaprofitar la xarxa

ResNet18 no sap què és el gest de «pedra». **Però ha après moltes altres coses:** vores, textures, corbes, parts
d'objectes. És la idea que ens acompanya des de la UT5: **les capes ocultes aprenen les característiques**. Només
l'última capa, la que decideix entre les 1.000 classes, és específica d'ImageNet.

Per tant, ens en quedem tot menys l'última capa, i en posem una de pròpia. Això és el **transfer learning**:

| Pas | Què entra | Què surt | Qui ho fa |
|---|---|---|---|
| 1. Extreure característiques | Una imatge de 224 × 224 | 512 números | ResNet18, sense entrenar res |
| 2. Decidir | Els 512 números | La classe | Un `MLPClassifier`, com els de la UT5 |

**Per què és útil?** Els 512 números ja descriuen la imatge. L'MLP només ha d'aprendre a combinar-los: en lloc
d'11 milions de pesos, amb 64 neurones n'entrena menys de 34.000. I com que el pas 1 només es fa **una vegada**,
després ja no cal tornar a passar les imatges per ResNet18.

### Les dades: pedra, paper, tisora

Són 450 imatges d'entrenament i 210 de test, amb tres classes. Fixa't que estan **generades per ordinador**, sobre
fons blanc: és important per a més endavant. Amb tan poques imatges no en separem una part més, i el test ens farà
de referència per comparar. Com que `load_image_zip` retorna un diccionari, les separem.

In [ ]:
rps = load_image_zip(URL_RPS)
X_train, y_train, rps_names = rps["train"]
X_test, y_test, _ = rps["test"]

print("Entrenament:", tuple(X_train.shape), "· Test:", tuple(X_test.shape))
print("Classes:", rps_names)

fig, grafics = plt.subplots(1, 6, figsize=(9, 1.8))
for grafic, position in zip(grafics, range(0, 450, 75)):
    grafic.imshow(X_train[position].permute(1, 2, 0).numpy())
    grafic.set_title(rps_names[y_train[position]], fontsize=9)
    grafic.axis("off")
plt.show()

I ara, el pas 1, una sola vegada. La funció `extract_features` passa les imatges per ResNet18, sense l'última capa.

> ⏱ Uns 20 segons a un ordinador normal, i un minut al CPU de Colab.

In [ ]:
start = time.perf_counter()
F_train = extract_features(X_train)
F_test = extract_features(X_test)
print("Característiques:", F_train.shape, f"· {time.perf_counter() - start:.0f} s")
print("Els primers 5 números de la primera imatge:", F_train[0][:5].round(2))

Ja no hi ha imatges: per a cadascuna tenim **512 números**. Ara són dades com les dels pingüins de la UT5, una fila
per mostra, i `MLPClassifier` les pot aprendre.

### Ara tu · Missió 2: entrena'n l'última capa

1. **Aposta.** Quin encert de test creus que traurà un MLP entrenat només amb aquestes 450 files? Quant creus que
   tardarà a entrenar-se?
2. **Experimenta.** Entrena `MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=SEED)` amb
   `F_train` i `y_train` (cal convertir-la amb `.numpy()`) i mesura el temps. Calcula'n l'encert amb `F_test`.
3. **Compta** els pesos que ha entrenat: 512 entrades × 64, més 64 × 3, més els biaixos. Compara-ho amb els 11 milions
   de ResNet18.
4. **Explica.** D'on ve tanta rapidesa? Quina part del temps s'ha gastat a extreure característiques, i quina a
   entrenar? Si afegissis una quarta classe, què hauries de tornar a fer i què no?

In [ ]:
# El teu codi aquí

## 3. Pocs exemples

Fins aquí, 150 imatges per classe. **I si en tinguéssim moltes menys?** Al NB 1.2 ho vam mesurar amb MNIST i
entrenant des de zero: un 72% amb 100 exemples i un 94% amb 1.000.

La funció `take_per_class(y_train, n)` tria `n` imatges de cada classe i en retorna les posicions. Hi ha dues maneres
de resoldre-ho, i aquí les posarem a competir:

- **Transfer learning:** `MLPClassifier` amb `F_train[positions]` i `y_train[positions].numpy()`.
- **CNN des de zero:** `make_cnn(3, 3, image_size=64, filters=(16, 32, 64), batch_norm=True)`, 20 epochs. Perquè
  vagi de pressa, amb imatges de 64 × 64.

La cel·la següent prepara les imatges petites.

In [ ]:
rps_small = load_image_zip(URL_RPS, size=64)
S_train = rps_small["train"][0]
S_test = rps_small["test"][0]
print(tuple(S_train.shape))

### Ara tu · Missió 3: amb quantes imatges?

1. **Aposta.** Amb 5 imatges per classe, quin encert traurà cadascuna? I a partir de quantes imatges creus que la CNN
   des de zero atraparà el transfer learning? Ho farà mai?
2. **Experimenta.** Per a `n` = 5, 10, 20, 50, 100 i 150, entrena els dos models **amb les mateixes imatges**. Per a
   la CNN: `make_loaders(TensorDataset(S_train[positions], y_train[positions]), TensorDataset(S_test, y_test),
   batch_size=16)` i `lr=0.001`.
3. **Dibuixa** l'encert de test dels dos models segons `n`.
4. **Explica.** En quin punt canvia la història? Si tinguessis un milió d'imatges, quin triaries? I si en tens 30?
   Recorda aquell NB 1.2: el transfer learning fa que hagis de buscar menys dades.

> ⏱ Dels sis entrenaments de la CNN, el més llarg tarda uns 15 segons a un ordinador normal. Els de l'MLP, res.

In [ ]:
# El teu codi aquí

## 4. La teva mà

Aquestes mans són generades per ordinador i sobre fons blanc. **I amb una mà de veritat?** Ara ho sabrem.

La norma d'abans continua: **a la foto només hi pot sortir la mà, sense la cara, cap altra persona ni dades privades.**

A Colab, `upload_photo()` obre un selector i retorna el nom del fitxer. `load_photo` la llegeix, i gira la imatge si
el mòbil la va fer de costat. `predict_photo(mlp, photo, rps_names)` la passa per ResNet18 i pel teu MLP.

```python
photo = load_photo(upload_photo())
show_prediction(photo, predict_photo(mlp, photo, rps_names))
```

Si la foto és HEIC, d'iPhone, `load_photo` t'explica com convertir-la. Fora de Colab, escriu directament la ruta:
`load_photo("la_meva_ma.jpg")`.

### Ara tu · Missió 4: la realitat

1. **Aposta.** Quantes de 6 fotos de la teva mà creus que encertarà? Apunta-ho abans de provar res.
2. **Fes 6 fotos:** pedra, paper i tisora, primer amb una paret o un paper blanc al darrere, i després amb un fons
   qualsevol, sense res privat.
3. **Prova-les** amb el millor MLP de la missió 2 i apunta els resultats en una taula: gest, fons, resposta i
   probabilitat.
4. **Explica.** Has encertat l'aposta? Quina diferència hi ha entre el test d'abans i aquestes fotos? Què vol dir
   això sobre el número que es pot prometre a un client? Com ho arreglaries? **És el que faràs al projecte del NB 6.6.**

In [ ]:
# El teu codi aquí

## 5. Un altre món

ImageNet són fotos fetes des del terra: gats, cafès, cotxes. **Serveix el que ha après per a una cosa que no s'hi
assembla gens?** Provem-ho amb **EuroSAT**: fotos de satèl·lit Sentinel-2 de 64 × 64 píxels, amb 10 tipus de terreny,
80 d'entrenament i 20 de test per classe.

La cel·la següent carrega les imatges i les ensenya.

In [ ]:
eurosat = load_image_zip(URL_EUROSAT)
X_sat, y_sat, sat_names = eurosat["train"]
X_sat_test, y_sat_test, _ = eurosat["test"]

fig, grafics = plt.subplots(2, 5, figsize=(9, 4))
for number, grafic in enumerate(grafics.flat):
    grafic.imshow(X_sat[y_sat == number][0].permute(1, 2, 0).numpy())
    grafic.set_title(sat_names[number], fontsize=9)
    grafic.axis("off")
plt.show()

### Ara tu · Missió 5: des de l'espai

1. **Aposta.** Quina classe d'ImageNet dirà ResNet18 d'una imatge de bosc vista des de l'espai? Amb característiques
   d'ImageNet i 80 imatges per classe, quin encert traurà l'MLP? Guanyarà la CNN des de zero?
2. **Mira què en diu** ResNet18 amb `top3_imagenet` per a dues imatges.
3. **Experimenta.** Extreu les característiques amb `extract_features` (uns 30 segons), entrena l'MLP i calcula'n
   l'encert. Carrega les imatges amb `load_image_zip(URL_EUROSAT, size=64)` i entrena una CNN amb la mateixa
   configuració de la missió 3, ara amb 10 classes.
4. **Explica.** Si ResNet18 no sap què és un conreu, per què les seves característiques ajuden a distingir-los?
   Quines classes confon? Pots fer servir la funció `predict` de l'MLP i comparar-la amb les etiquetes.

> ⏱ L'entrenament de la CNN tarda uns 30 segons a un ordinador normal.

In [ ]:
# El teu codi aquí

## 6. Resum

| | CNN des de zero | Transfer learning |
|---|---|---|
| **Què s'entrena** | Tota la xarxa | Només la part final |
| **Temps** | Segons o minuts | Una extracció i menys d'un segon |
| **Amb poques imatges** | Fluixa | Molt millor |
| **Amb moltes imatges** | Pot atrapar-lo | Es queda limitat al que ja sap la xarxa |

### Les idees que t'has d'endur

1. **Les capes ocultes aprenen característiques, i aquestes es poden reaprofitar.** Això és el transfer learning.
2. **El test només mesura el que s'assembla al test.** Si el món real no s'hi assembla, el número no val.
3. **Abans d'entrenar res, pregunta't si algú ja ho ha fet.** Sovint és la manera més barata de començar.

## Exercicis finals

Les cinc missions les has fet pel camí. Aquests dos exercicis et demanen tot el notebook.

**Exercici 6. Quines errades?** Busca a `X_test` les imatges de pedra-paper-tisora en què s'equivoca el teu MLP i
dibuixa'n algunes amb la predicció i la classe real. Hi ha algun patró? Fes servir també `top3_imagenet` amb elles: què
veu ResNet18 en aquestes imatges?

**Exercici 7. Fine-tuning (només amb GPU).** Fins ara, ResNet18 no s'ha mogut. El **fine-tuning** deixa que l'últim
bloc de capes també aprengui, amb molta cura. A Colab, activa la GPU (Entorn d'execució → Canvia el tipus d'entorn),
crea la xarxa amb `make_pretrained(3, unfreeze=True)` i entrena-la amb `train_model` 5 epochs, `lr=0.0001` i
`batch_size=16`, amb `X_train` i `X_test`. Compara-la amb `make_pretrained(3, unfreeze=False)` i amb el teu MLP.
Compta els pesos de les dues amb `count_parameters`. Val la pena? Sense GPU, cada epoch tarda més d'un minut i cal paciència.

## Per al debat de classe

1. **Herència.** Una xarxa preentrenada porta els biaixos de les seves dades: fotos sobretot d'internet i d'alguns
   països. Què pot passar si l'aprofites per a un problema de casa nostra?
2. **Dependència.** Si tot el món parteix de les mateixes poques xarxes, quins avantatges té i quins riscos?
3. **Promeses.** Un fabricant diu que el seu model arriba al 99%. Què li preguntaries, després d'aquesta missió 4?